# 📘 Capítulo 8 — Autovalores y Autovectores

## 🎯 ¿Qué aprenderemos?

Imagina que tienes una transformación lineal $T$ representada por una matriz $A$.  
La mayoría de los vectores **cambian de dirección** cuando se multiplican por $A$.  
Pero hay ciertos vectores **muy especiales** que **solo se escalan**: su dirección no cambia.

Esos vectores especiales se llaman **autovectores** y los factores de escala se llaman **autovalores**.

### 🎯 Objetivos de esta sección
1. Entender la definición algebraica y el significado geométrico de autovalores y autovectores.
2. Aprender a calcularlos con el **polinomio característico**.
3. Comprender la **diagonalización** $A = PDP^{-1}$ y sus aplicaciones.
4. Implementar todo en Python con `numpy` y `sympy`.

## 🟦 Definición: Autovalores y Autovectores

Sea $A$ una matriz cuadrada de orden $n \times n$. Decimos que:

- $\lambda \in \mathbb{R}$ (o $\mathbb{C}$) es un **autovalor** de $A$ si existe un vector **no nulo** $\mathbf{v}$ tal que:

$$A\mathbf{v} = \lambda\mathbf{v}$$

- El vector $\mathbf{v} \neq \mathbf{0}$ que satisface esta ecuación se llama **autovector** asociado al autovalor $\lambda$.

### 💡 Significado geométrico

La ecuación $A\mathbf{v} = \lambda\mathbf{v}$ dice que la transformación $A$ **no rota** el vector $\mathbf{v}$, solo lo **escala** por el factor $\lambda$:

| Valor de $\lambda$ | Efecto sobre $\mathbf{v}$ |
|:---|:---|
| $\lambda > 1$ | El vector se **estira** (mismo sentido) |
| $0 < \lambda < 1$ | El vector se **comprime** (mismo sentido) |
| $\lambda = 1$ | El vector **no cambia** |
| $\lambda < 0$ | El vector se **invierte de sentido** y escala |
| $\lambda = 0$ | El vector se **colapsa** al origen |

### 🔍 ¿Cómo encontrarlos?

Reescribimos $A\mathbf{v} = \lambda\mathbf{v}$ como:

$$(A - \lambda I)\mathbf{v} = \mathbf{0}$$

Para que exista solución no trivial ($\mathbf{v} \neq \mathbf{0}$), la matriz $(A - \lambda I)$ debe ser **singular**, es decir:

$$\det(A - \lambda I) = 0 \quad \leftarrow \text{Ecuación característica}$$

Esta ecuación en $\lambda$ se llama el **polinomio característico** de $A$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definimos la matriz de transformación
A = np.array([[3.0, 1.0],
              [0.0, 2.0]])

# Generamos muchos vectores unitarios en todas las direcciones
angles = np.linspace(0, 2*np.pi, 36, endpoint=False)
vectors = np.array([[np.cos(a), np.sin(a)] for a in angles])

# Aplicamos la transformación a cada vector
transformed = (A @ vectors.T).T

# Calculamos los autovectores reales con numpy
eigenvalues, eigenvectors = np.linalg.eig(A)

# Detectamos qué vectores mantienen su dirección (cos(ángulo) ≈ ±1)
cosines = np.array([
    abs(np.dot(v, t) / (np.linalg.norm(v) * np.linalg.norm(t) + 1e-10))
    for v, t in zip(vectors, transformed)
])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Izquierda: todos los vectores originales y transformados
ax = axes[0]
for i, (v, t) in enumerate(zip(vectors, transformed)):
    color = 'red' if cosines[i] > 0.99 else 'lightblue'
    lw = 2.5 if cosines[i] > 0.99 else 0.8
    ax.annotate('', xy=v, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    ax.annotate('', xy=t*0.5, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5, alpha=0.4))
ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_aspect('equal')
ax.set_title('Vectores originales (azul) y transformados (gris)\n'
             'Los rojos mantienen dirección → ¡Autovectores!', fontsize=11)
ax.set_xlabel('x'); ax.set_ylabel('y')

# Derecha: autovectores resaltados
ax2 = axes[1]
circle = plt.Circle((0,0), 1, fill=False, color='gray', lw=1, ls='--')
ax2.add_patch(circle)
colors_ev = ['#e74c3c', '#3498db']
for i, (lam, ev) in enumerate(zip(eigenvalues, eigenvectors.T)):
    ev_norm = ev / np.linalg.norm(ev)
    ax2.annotate('', xy=ev_norm, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color=colors_ev[i], lw=2.5))
    ax2.annotate('', xy=lam*ev_norm*0.6, xytext=(0,0),
                 arrowprops=dict(arrowstyle='->', color=colors_ev[i], lw=1.5,
                                 linestyle='dashed'))
    ax2.text(ev_norm[0]*1.1, ev_norm[1]*1.1,
             f'$\\mathbf{{v}}_{i+1}$, $\\lambda_{i+1}={lam:.1f}$',
             color=colors_ev[i], fontsize=12)
ax2.set_xlim(-2, 2); ax2.set_ylim(-2, 2)
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(0, color='k', lw=0.5)
ax2.set_aspect('equal')
ax2.set_title('Autovectores de $A = \\begin{pmatrix}3&1\\\\0&2\\end{pmatrix}$\n'
              'La flecha discontinua muestra el vector transformado', fontsize=11)
ax2.set_xlabel('x'); ax2.set_ylabel('y')

plt.tight_layout()
plt.savefig('autovectores_demo.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Autovalores: {eigenvalues}')
print(f'Autovectores (columnas):\n{eigenvectors}')

## 🟩 Interpretación Geométrica

Los autovectores definen **líneas invariantes** bajo la transformación:
cualquier punto sobre la línea spanned por un autovector permanece en esa misma línea después de aplicar $A$.

### 📐 Efectos según el signo y magnitud de $\lambda$

```
λ > 1 :   ──────►  ══════════════►   (estiramiento)
0<λ<1 :   ──────►  ──►              (compresión)
λ = -1:   ──────►  ◄──────           (reflexión)
λ = 0 :   ──────►  •                 (colapso al origen)
```

### 🔑 Ideas clave

1. **Los autovectores no son únicos**: si $\mathbf{v}$ es autovector, también lo es $c\mathbf{v}$ para cualquier $c \neq 0$.  
   Por eso siempre hablamos de la **dirección** del autovector.

2. **El subespacio propio** $E_\lambda = \ker(A - \lambda I)$ contiene todos los autovectores de $\lambda$ más el vector cero.

3. **Multiplicidad algebraica** vs **multiplicidad geométrica**:  
   - Algebraica: multiplicidad del autovalor como raíz del polinomio característico.  
   - Geométrica: dimensión del subespacio propio $\dim(\ker(A-\lambda I))$.

4. **Traza y determinante**: para una matriz $n\times n$,
$$\text{tr}(A) = \sum_{i=1}^n \lambda_i, \qquad \det(A) = \prod_{i=1}^n \lambda_i$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

def plot_transformation(A, title, ax):
    """Visualiza cómo A transforma el círculo unitario"""
    t = np.linspace(0, 2*np.pi, 300)
    circle = np.vstack([np.cos(t), np.sin(t)])  # 2 x 300
    ellipse = A @ circle  # imagen del círculo

    eigenvalues, eigenvectors = np.linalg.eig(A)

    ax.plot(circle[0], circle[1], 'b--', lw=1, alpha=0.5, label='Círculo original')
    ax.plot(ellipse[0], ellipse[1], 'g-', lw=2, label='Imagen (elipse)')

    colors = ['#e74c3c', '#9b59b6']
    for i, (lam, ev) in enumerate(zip(eigenvalues, eigenvectors.T)):
        if np.isreal(lam):
            ev_r = ev.real
            ev_norm = ev_r / np.linalg.norm(ev_r)
            scale = float(np.real(lam))
            # Autovector original
            ax.annotate('', xy=ev_norm, xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
            # Autovector transformado = λ * v
            ax.annotate('', xy=scale*ev_norm, xytext=(0,0),
                        arrowprops=dict(arrowstyle='->', color=colors[i], lw=2,
                                        linestyle='dashed', alpha=0.7))
            ax.text(ev_norm[0]*1.15, ev_norm[1]*1.15,
                    f'$\\lambda={lam:.2f}$', color=colors[i], fontsize=10,
                    ha='center')

    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_aspect('equal')
    ax.legend(fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Estiramiento puro (diagonal)
A1 = np.array([[3.0, 0.0], [0.0, 1.5]])
plot_transformation(A1, 'Estiramiento diagonal\n$A = \\mathrm{diag}(3, 1.5)$', axes[0])

# Matriz con autovectores oblicuos
A2 = np.array([[3.0, 1.0], [0.0, 2.0]])
plot_transformation(A2, 'Matriz triangular\n$A=\\begin{pmatrix}3&1\\\\0&2\\end{pmatrix}$', axes[1])

# Matriz simétrica
A3 = np.array([[2.0, 1.0], [1.0, 2.0]])
plot_transformation(A3, 'Matriz simétrica\n$A=\\begin{pmatrix}2&1\\\\1&2\\end{pmatrix}$', axes[2])

plt.tight_layout()
plt.savefig('transformacion_circulo.png', dpi=100, bbox_inches='tight')
plt.show()
print('Los autovectores (flechas) muestran las direcciones invariantes de cada transformación.')

## 🟨 El Polinomio Característico

La ecuación $\det(A - \lambda I) = 0$ genera un **polinomio** en $\lambda$ de grado $n$ (para una matriz $n\times n$).

### Para una matriz $2 \times 2$:

$$A = \begin{pmatrix} a & b \\ c & d \end{pmatrix}$$

$$\det(A - \lambda I) = \det\begin{pmatrix} a-\lambda & b \\ c & d-\lambda \end{pmatrix} = (a-\lambda)(d-\lambda) - bc$$

$$= \lambda^2 - (a+d)\lambda + (ad-bc) = \lambda^2 - \text{tr}(A)\lambda + \det(A) = 0$$

### Para una matriz $3 \times 3$:

$$A = \begin{pmatrix} a & b & c \\ d & e & f \\ g & h & i \end{pmatrix}$$

$$\det(A - \lambda I) = -\lambda^3 + \text{tr}(A)\lambda^2 - M_2\lambda + \det(A)$$

donde $M_2$ es la suma de los menores principales de orden 2.

### 🔑 Propiedades del polinomio característico

| Propiedad | Expresión |
|:---|:---|
| Grado del polinomio | $n$ (igual al tamaño de la matriz) |
| Número de autovalores (con multiplicidad) | $n$ (sobre $\mathbb{C}$) |
| Coeficiente líder | $(-1)^n$ |
| Término independiente | $\det(A)$ (salvo signo) |
| Coeficiente de $\lambda^{n-1}$ | $-\text{tr}(A)$ (salvo signo) |

### 📌 Procedimiento para calcular autovalores y autovectores

1. **Calcular** $p(\lambda) = \det(A - \lambda I)$.
2. **Resolver** $p(\lambda) = 0$ para obtener los autovalores $\lambda_1, \lambda_2, \ldots$
3. **Para cada $\lambda_i$**: resolver el sistema $(A - \lambda_i I)\mathbf{v} = \mathbf{0}$ para encontrar los autovectores.

In [ ]:
import sympy as sp

lam = sp.Symbol('lambda')

print('='*60)
print('EJEMPLO 1: Matriz 2x2')
print('='*60)

A2 = sp.Matrix([[3, 1], [0, 2]])
print(f'A =\n{A2}\n')

# Polinomio característico
char_poly_2 = (A2 - lam*sp.eye(2)).det()
char_poly_2_expanded = sp.expand(char_poly_2)
print(f'det(A - λI) = {char_poly_2_expanded}')

roots_2 = sp.solve(char_poly_2_expanded, lam)
print(f'Autovalores: λ = {roots_2}')

# Autovectores
for root in roots_2:
    M = A2 - root*sp.eye(2)
    null = M.nullspace()
    print(f'  Para λ = {root}: autovector(es) = {null}')

print()
print('='*60)
print('EJEMPLO 2: Matriz 3x3')
print('='*60)

A3 = sp.Matrix([[2, 1, 0],
                [1, 3, 1],
                [0, 1, 2]])
print(f'A =\n{A3}\n')

char_poly_3 = (A3 - lam*sp.eye(3)).det()
char_poly_3_exp = sp.expand(char_poly_3)
print(f'Polinomio característico: {char_poly_3_exp}')

roots_3 = sp.solve(char_poly_3_exp, lam)
print(f'Autovalores: λ = {roots_3}')

for root in roots_3:
    M = A3 - root*sp.eye(3)
    null = M.nullspace()
    print(f'  Para λ = {root}: autovector = {null[0].T}')

# Verificación: traza = suma de autovalores
print()
print('Verificación:')
print(f'  tr(A) = {A3.trace()} = suma de autovalores = {sum(roots_3)}')
print(f'  det(A) = {A3.det()} = producto de autovalores = {sp.prod(roots_3)}')

## 🔎 Cómo Encontrar Autovectores — Paso a Paso

Una vez que tenemos los autovalores $\lambda_1, \lambda_2, \ldots$, encontramos los autovectores resolviendo:

$$(A - \lambda_i I)\mathbf{v} = \mathbf{0}$$

Esto es equivalente a encontrar el **núcleo** (kernel) de la matriz $A - \lambda_i I$.

### 📋 Procedimiento detallado para $2 \times 2$

Sea $A = \begin{pmatrix} 3 & 1 \\ 0 & 2 \end{pmatrix}$ con autovalores $\lambda_1 = 3$, $\lambda_2 = 2$.

**Para $\lambda_1 = 3$:**
$$A - 3I = \begin{pmatrix} 0 & 1 \\ 0 & -1 \end{pmatrix} \xrightarrow{\text{RREF}} \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}$$

El sistema es: $v_2 = 0$, con $v_1$ libre.  
Por tanto: $\mathbf{v}_1 = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$ (o cualquier múltiplo no nulo).

**Para $\lambda_2 = 2$:**
$$A - 2I = \begin{pmatrix} 1 & 1 \\ 0 & 0 \end{pmatrix}$$

El sistema es: $v_1 + v_2 = 0$, con $v_2$ libre.  
Tomando $v_2 = 1$: $\mathbf{v}_2 = \begin{pmatrix} -1 \\ 1 \end{pmatrix}$.

### ✅ Verificación
$$A\mathbf{v}_1 = \begin{pmatrix}3&1\\0&2\end{pmatrix}\begin{pmatrix}1\\0\end{pmatrix} = \begin{pmatrix}3\\0\end{pmatrix} = 3\begin{pmatrix}1\\0\end{pmatrix} = \lambda_1 \mathbf{v}_1 \checkmark$$

$$A\mathbf{v}_2 = \begin{pmatrix}3&1\\0&2\end{pmatrix}\begin{pmatrix}-1\\1\end{pmatrix} = \begin{pmatrix}-2\\2\end{pmatrix} = 2\begin{pmatrix}-1\\1\end{pmatrix} = \lambda_2 \mathbf{v}_2 \checkmark$$

In [ ]:
import sympy as sp
import numpy as np

print('EJEMPLO COMPLETO PASO A PASO')
print('='*55)
print('Matriz: A = [[4, 2], [1, 3]]')
print()

A = sp.Matrix([[4, 2], [1, 3]])
lam = sp.Symbol('lambda')

# Paso 1: Matriz característica
print('PASO 1: Calcular A - λI')
AminusLI = A - lam * sp.eye(2)
print(f'A - λI = {AminusLI}')

# Paso 2: Determinante
print()
print('PASO 2: Calcular det(A - λI)')
p = sp.expand(AminusLI.det())
print(f'p(λ) = det(A - λI) = {p}')
print(f'     = λ² - tr(A)λ + det(A)')
print(f'     = λ² - {A.trace()}λ + {A.det()}')

# Paso 3: Raíces
print()
print('PASO 3: Resolver p(λ) = 0')
roots = sp.solve(p, lam)
print(f'Autovalores: λ₁ = {roots[0]}, λ₂ = {roots[1]}')

# Paso 4: Autovectores
print()
print('PASO 4: Encontrar autovectores')
eigenvecs = []
for i, root in enumerate(roots):
    M = A - root * sp.eye(2)
    print(f'\n  Para λ = {root}:')
    print(f'  A - {root}I = {M}')
    null = M.nullspace()
    ev = null[0]
    eigenvecs.append(ev)
    print(f'  Autovector: v = {ev.T}')

# Paso 5: Verificación
print()
print('PASO 5: Verificación A*v = λ*v')
for i, (root, ev) in enumerate(zip(roots, eigenvecs)):
    Av = A * ev
    lv = root * ev
    ok = '✅' if Av == lv else '❌'
    print(f'  λ = {root}: A*v = {Av.T}, λ*v = {lv.T} {ok}')

# Paso 6: Resumen
print()
print('RESUMEN:')
print(f'  tr(A) = {A.trace()} = {roots[0]} + {roots[1]} = {roots[0]+roots[1]} ✅')
print(f'  det(A) = {A.det()} = {roots[0]} × {roots[1]} = {roots[0]*roots[1]} ✅')

## 🟦 Diagonalización: $A = PDP^{-1}$

Una de las aplicaciones más poderosas de los autovectores es la **diagonalización** de matrices.

### �� Definición

Una matriz $A$ es **diagonalizable** si existe una matriz invertible $P$ tal que:

$$A = PDP^{-1}$$

donde $D$ es una **matriz diagonal** cuyas entradas son los autovalores de $A$:

$$D = \begin{pmatrix} \lambda_1 & 0 & \cdots & 0 \\ 0 & \lambda_2 & \cdots & 0 \\ \vdots & & \ddots & \vdots \\ 0 & 0 & \cdots & \lambda_n \end{pmatrix}$$

y $P$ es la matriz cuyas **columnas son los autovectores** correspondientes:

$$P = \begin{pmatrix} | & | & & | \\ \mathbf{v}_1 & \mathbf{v}_2 & \cdots & \mathbf{v}_n \\ | & | & & | \end{pmatrix}$$

### ✅ Condición de diagonalizabilidad

$A$ es diagonalizable **si y solo si** tiene $n$ autovectores linealmente independientes, lo que ocurre cuando:
- $A$ tiene $n$ autovalores distintos, **o**
- Para cada autovalor, la multiplicidad geométrica = multiplicidad algebraica.

### 🎯 ¿Por qué es útil?

Si $A = PDP^{-1}$, entonces:
$$A^k = PD^kP^{-1} \quad \text{donde} \quad D^k = \begin{pmatrix} \lambda_1^k & & \\ & \ddots & \\ & & \lambda_n^k \end{pmatrix}$$

¡Calcular $A^{100}$ se reduce a elevar los autovalores a la potencia 100!

In [ ]:
import numpy as np

print('DIAGONALIZACIÓN con numpy')
print('='*55)

A = np.array([[4.0, 2.0],
              [1.0, 3.0]])
print(f'A =\n{A}\n')

# Obtener autovalores y autovectores
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f'Autovalores (diagonal de D):')
print(f'  λ = {eigenvalues}')
print(f'\nAutovectores (columnas de P):')
print(eigenvectors)

# Construir P y D
P = eigenvectors          # columnas = autovectores
D = np.diag(eigenvalues)  # diagonal = autovalores
P_inv = np.linalg.inv(P)

print(f'\nMatriz P (autovectores como columnas):\n{P}')
print(f'\nMatriz D (diagonal de autovalores):\n{D}')

# Verificación: A = P D P^{-1}
A_reconstructed = P @ D @ P_inv
print(f'\nVerificación: P @ D @ inv(P) =\n{np.round(A_reconstructed, 10)}')
print(f'¿A = PDP⁻¹? {np.allclose(A, A_reconstructed)}')

# Verificación individual: A*vi = λi*vi
print('\nVerificación A*v = λ*v para cada autovector:')
for i in range(len(eigenvalues)):
    vi = eigenvectors[:, i]
    Avi = A @ vi
    lvi = eigenvalues[i] * vi
    ok = '✅' if np.allclose(Avi, lvi) else '❌'
    print(f'  λ = {eigenvalues[i]:.1f}: A·v = {np.round(Avi,4)}, λ·v = {np.round(lvi,4)} {ok}')

## 🧮 Propiedades Importantes

### 📊 Tabla de propiedades clave

| Propiedad | Enunciado | Utilidad |
|:---|:---|:---|
| **Traza** | $\text{tr}(A) = \sum_{i=1}^n \lambda_i$ | Verificación rápida |
| **Determinante** | $\det(A) = \prod_{i=1}^n \lambda_i$ | Verificación rápida |
| **Inversibilidad** | $A$ invertible $\Leftrightarrow$ $0$ no es autovalor | Importante en sistemas |
| **Matrices simétricas** | Siempre diagonalizables con autovectores **ortogonales** | Espectral theorem |
| **Potencias** | $A^k = PD^kP^{-1}$ si $A$ es diagonalizable | Sistemas dinámicos |
| **Invertibilidad de $A$** | $(A^{-1})$ tiene autovalores $1/\lambda_i$ | Inversa sin calcular |
| **Polinomio de $A$** | $p(A)$ tiene autovalores $p(\lambda_i)$ | Cayley-Hamilton |

### 🔑 Teorema Espectral (matrices simétricas)

> Si $A$ es **simétrica** ($A = A^T$), entonces:
> 1. Todos los autovalores son **reales**.
> 2. Los autovectores de autovalores distintos son **ortogonales**.
> 3. $A$ es **ortogonalmente diagonalizable**: $A = QDQ^T$ con $Q$ ortogonal.

### 💡 Autovalores de matrices especiales

| Tipo de matriz | Autovalores |
|:---|:---|
| Diagonal | Las entradas de la diagonal |
| Triangular | Las entradas de la diagonal |
| Ortogonal | Magnitud = 1 (sobre $\mathbb{C}$) |
| Proyección | Solo 0 y 1 |
| Idempotente ($A^2=A$) | Solo 0 y 1 |
| Nilpotente ($A^k=0$) | Solo 0 |

In [ ]:
import numpy as np

print('VERIFICACIÓN: tr(A) = Σλᵢ  y  det(A) = Πλᵢ')
print('='*55)

matrices = {
    '2x2 general': np.array([[4.0, 2.0], [1.0, 3.0]]),
    '2x2 simétrica': np.array([[5.0, 3.0], [3.0, 2.0]]),
    '3x3 triangular': np.array([[2.0,1.0,0.0],[0.0,3.0,2.0],[0.0,0.0,5.0]]),
    '3x3 simétrica': np.array([[4.0,2.0,1.0],[2.0,3.0,1.0],[1.0,1.0,2.0]]),
}

for name, A in matrices.items():
    eigenvalues = np.linalg.eigvals(A)
    tr_A = np.trace(A)
    det_A = np.linalg.det(A)
    sum_lam = np.sum(eigenvalues)
    prod_lam = np.prod(eigenvalues)

    tr_ok = '✅' if np.isclose(tr_A, sum_lam.real) else '❌'
    det_ok = '✅' if np.isclose(det_A, prod_lam.real) else '❌'

    print(f'\nMatriz: {name}')
    print(f'  Autovalores: {np.round(eigenvalues.real, 4)}')
    print(f'  tr(A) = {tr_A:.4f}  |  Σλᵢ = {sum_lam.real:.4f}  {tr_ok}')
    print(f'  det(A) = {det_A:.4f}  |  Πλᵢ = {prod_lam.real:.4f}  {det_ok}')

## 🚀 El Poder de la Diagonalización: $A^k = PD^kP^{-1}$

### ¿Por qué es tan poderosa?

Calcular $A^{100}$ directamente requeriría 99 multiplicaciones de matrices.  
Con diagonalización: **solo 3 operaciones**.

$$A^k = (PDP^{-1})^k = P\underbrace{(D \cdots D)}_{k \text{ veces}}P^{-1} = PD^kP^{-1}$$

Y $D^k$ es trivial:
$$D^k = \begin{pmatrix} \lambda_1^k & 0 \\ 0 & \lambda_2^k \end{pmatrix}$$

### 📐 Aplicación: Modelos de población

Sea $\mathbf{x}(t+1) = A\mathbf{x}(t)$ un sistema dinámico lineal (ej: evolución de edades en una población).

$$\mathbf{x}(t) = A^t\mathbf{x}(0) = PD^tP^{-1}\mathbf{x}(0)$$

El **comportamiento a largo plazo** está dominado por el autovalor de **mayor módulo** (radio espectral).

### 📌 Sucesión de Fibonacci

La sucesión de Fibonacci $F_{n+2} = F_{n+1} + F_n$ puede escribirse como:
$$\begin{pmatrix} F_{n+1} \\ F_n \end{pmatrix} = A^n \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad A = \begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix}$$

Los autovalores de $A$ son $\phi = \frac{1+\sqrt{5}}{2} \approx 1.618$ (número áureo) y $\hat{\phi} = \frac{1-\sqrt{5}}{2}$.

In [ ]:
import numpy as np
import time

A = np.array([[4.0, 2.0],
              [1.0, 3.0]])

eigenvalues, P = np.linalg.eig(A)
P_inv = np.linalg.inv(P)

# ── Método 1: Multiplicación directa ──────────────────────
k = 20
t0 = time.perf_counter()
A_power_direct = np.eye(2)
for _ in range(k):
    A_power_direct = A_power_direct @ A
t1 = time.perf_counter()

# ── Método 2: Diagonalización ──────────────────────────────
t2 = time.perf_counter()
D_k = np.diag(eigenvalues**k)          # D^k: solo potencias escalares
A_power_diag = P @ D_k @ P_inv         # PD^kP^{-1}
t3 = time.perf_counter()

print(f'A^{k} mediante multiplicación directa:')
print(np.round(A_power_direct))
print(f'Tiempo: {(t1-t0)*1e6:.2f} μs')

print(f'\nA^{k} mediante diagonalización (PD^kP⁻¹):')
print(np.round(A_power_diag.real))
print(f'Tiempo: {(t3-t2)*1e6:.2f} μs')

print(f'\n¿Son iguales? {np.allclose(A_power_direct, A_power_diag.real, atol=1e-6)}')

# ── Fibonacci con diagonalización ─────────────────────────
print('\n' + '='*55)
print('FIBONACCI con diagonalización')
Af = np.array([[1.0, 1.0], [1.0, 0.0]])
lam_f, Pf = np.linalg.eig(Af)
Pf_inv = np.linalg.inv(Pf)

print(f'Autovalores de la matriz de Fibonacci:')
print(f'  φ₁ = {lam_f[0]:.6f}  (número áureo = {(1+5**0.5)/2:.6f})')
print(f'  φ₂ = {lam_f[1]:.6f}')

x0 = np.array([1.0, 0.0])  # F_1 = 1, F_0 = 0
print(f'\nPrimeros 10 números de Fibonacci:')
for n in range(1, 11):
    Dfib = np.diag(lam_f**n)
    xn = Pf @ Dfib @ Pf_inv @ x0
    print(f'  F_{n:2d} = {int(round(xn[1].real))}')

## 📚 Ejercicios Propuestos

### 🧮 Ejercicio 1
Sea $A = \begin{pmatrix} 5 & 4 \\ 1 & 2 \end{pmatrix}$.  
(a) Calcula el polinomio característico.  
(b) Encuentra los autovalores.  
(c) Para cada autovalor, encuentra el autovector correspondiente.  
(d) Verifica que $A = PDP^{-1}$.

### 🧮 Ejercicio 2
Sea $A = \begin{pmatrix} 2 & 0 & 0 \\ 1 & 3 & 0 \\ 0 & 2 & 4 \end{pmatrix}$ (triangular inferior).  
¿Cuáles son sus autovalores sin calcular el determinante? Justifica.

### 🧮 Ejercicio 3
Demuestra que si $\lambda$ es autovalor de $A$ con autovector $\mathbf{v}$, entonces $\lambda^2$ es autovalor de $A^2$ con el mismo autovector.

### 🧮 Ejercicio 4
Sea $A = \begin{pmatrix} 0 & -1 \\ 1 & 0 \end{pmatrix}$ (rotación de 90°).  
(a) Calcula sus autovalores. ¿Son reales?  
(b) ¿Qué dice esto sobre si la rotación tiene direcciones invariantes?

### 🧮 Ejercicio 5
Sea $A = \begin{pmatrix} 3 & 0 \\ 0 & 3 \end{pmatrix}$ (múltiplo de la identidad).  
¿Cuántos autovectores linealmente independientes tiene? ¿Es diagonalizable?

---

## 💬 Conclusión

Los autovalores y autovectores son la **anatomía de una transformación lineal**: revelan sus
direcciones y escalas más naturales. Al diagonalizar una matriz, descomponemos la transformación
en sus partes más simples.

En el siguiente material exploraremos **cuatro ejemplos detallados** que solidificarán
tu comprensión geométrica y computacional.

➡️ Continúa con: [Ejemplos de Autovalores y Autovectores](ejemplos.ipynb)